In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from collections import Counter
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier

In [ ]:
d={}
b={}

df = pd.read_csv("final_extracted_3000data.csv")

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['grouped_diagnosa'])

text_cols = ['detailPemeriksaan', 'pengobatan', 'keluhan_utama_str', 'icd10_label']
df[text_cols] = df[text_cols].fillna('')

vectorizers = {}
X_parts = []

for col in text_cols:
    vectorizer = TfidfVectorizer(max_features=200)
    X_col = vectorizer.fit_transform(df[col])
    X_parts.append(X_col)
    vectorizers[col] = vectorizer

numeric_cols = ['age_days', 'beratBadan', 'tinggiBadan', 'nadi', 'suhu', 'pernapasan', 'SPO2', 'tekanan_sistolik', 'tekanan_diastolik']  
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df[numeric_cols] = df[numeric_cols].fillna(0)


scaler = StandardScaler()
X_num = scaler.fit_transform(df[numeric_cols])
X_parts.append(X_num)

cat_cols = ['pasien.jeniskelamin'] 
ohe = OneHotEncoder(handle_unknown='ignore')
X_cat = ohe.fit_transform(df[cat_cols].fillna(''))
X_parts.append(X_cat)

X = hstack(X_parts)


In [ ]:
def evaluate_model(model, model_name, X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels):
    pipeline = Pipeline([
        ('scaler', StandardScaler(with_mean=False)),  
        ('model', model) 
    ])
    
    pipeline.fit(X_train_resampled, y_train_resampled)
    y_pred = pipeline.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    rf_cv_scores = cross_val_score(pipeline, X_train_resampled, y_train_resampled, cv=stratified_cv, scoring='accuracy')
    
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, labels=test_labels, target_names=label_encoder.inverse_transform(test_labels)))
    print(f"K_fold: {rf_cv_scores}")
    
    d[model_name] = accuracy
    b[model_name] = rf_cv_scores

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from scipy import sparse

class_counts = Counter(y)
valid_classes = [cls for cls, count in class_counts.items() if count > 2]  
mask = np.isin(y, valid_classes)

X = sparse.csr_matrix(X) 
X_filtered = X[mask]
y_filtered = y[mask]


sss = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=42)
train_index, test_index = next(sss.split(X_filtered, y_filtered))
X_train, X_test = X_filtered[train_index], X_filtered[test_index]
y_train, y_test = y_filtered[train_index], y_filtered[test_index]

stratified_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

valid_classes = np.unique(y_train)
mask = np.isin(y_train_resampled, valid_classes)
X_train_resampled = X_train_resampled[mask]
y_train_resampled = y_train_resampled[mask]

test_labels = np.unique(y_test)

print("Random forest")
rf_model = RandomForestClassifier()
evaluate_model(rf_model, "rf", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)


print()
print("LightGBM")
lgb_model = LGBMClassifier()
evaluate_model(lgb_model, "lgb", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)


print()
print("SVC")
svc_model = SVC()
evaluate_model(svc_model, "svc", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)


print()
print("KNN")
knn_model = KNeighborsClassifier()
evaluate_model(knn_model, "knn", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)


print()
print("MLP")
mlp_model = MLPClassifier()
evaluate_model(mlp_model, "mlp", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)


print()
print("Logistic Regression")
lr_model = LogisticRegression()
evaluate_model(lr_model, "lr", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)

print()
print("XGBoost")
xgb_model = XGBClassifier()
evaluate_model(xgb_model, "xgb", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)

print()
print("Decision Tree")
destree_model = DecisionTreeClassifier()
evaluate_model(destree_model, "des_tree", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)

print()
print("CatBoost")
cat_model = CatBoostClassifier(learning_rate=0.1, iterations=100, depth=6, verbose=50, task_type='GPU')
evaluate_model(cat_model, "catboost", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)

print()
print("Gradient Boosting")
gb_model = GradientBoostingClassifier(verbose=1)
evaluate_model(gb_model, "gb", X_train_resampled, y_train_resampled, stratified_cv, X_test, y_test, test_labels)


In [ ]:
for ky,vl in b.items():
    print(f"{ky}: {vl}")
    print()

In [ ]:
for ky,vl in d.items():
    print(f"{ky}: {vl}")